# NeuroStore — GPU/TPU Benchmark & Simulation Lab

**Goal:** Evaluate the computational ideas in the user-supplied *NeuroStore: Empirical Evaluation of a Neural-Coding Data Architecture* on accelerators assigned by Google Colab.

This is an **independent, conceptual implementation**, not an execution of the report's original `coding.py`, `memory_sdm.py`, `memory_vsa.py`, `memory_hopfield.py`, or `benchmarks.py`. Results should not be described as reproducing the report until the original source code, seeds, datasets, and configurations have been reconciled.

## Experiments

1. Exact flat-vector retrieval versus Modern Hopfield softmax reconstruction and cleanup, under noisy queries.
2. FlyHash-style sparse random projection and winner-take-all similarity retrieval.
3. Role–filler VSA structured-record retrieval with ambiguous partial cues.
4. Kanerva-inspired sparse distributed memory (SDM) capacity and recall.
5. Out-of-distribution similarity rejection.
6. Simulated bit-flip and analog-like readout faults.
7. Device-resident and transfer-inclusive latency measurements, plots, and portable exports.

**Important distinctions**

- Colab allocates a particular CPU, GPU, or TPU runtime per session. Run the same notebook separately on each available runtime and combine exported CSVs.
- This notebook benchmarks **one JAX device** per session, even when a TPU runtime exposes several cores. It does not measure full-pod scaling.
- Hopfield softmax preserves the ranking of its input similarities. Any difference from exact retrieval here comes from the reconstruction-and-cleanup stage, not a fundamentally different first-stage nearest-neighbor search.
- FlyHash and VSA implementations are **algorithmic proxies** and may differ from the report's implementations.
- SDM uses a fixed number of nearest hard locations and vectorized construction rather than an exact recreation of the report's SDM implementation.
- Bit flips and additive perturbations are functional fault-injection tests, **not calibrated FeNAND, analog crossbar, or compute-in-memory device models**.
- Energy, joules/query, endurance, ADC/DAC overhead, and crossbar area are **not measured** by this notebook.
- JAX compilation and data transfers are excluded from device-resident steady-state timing unless explicitly noted. Batch latency is not individual request latency.

## Run instructions

1. In Google Colab, choose **Runtime → Change runtime type**, then select a GPU or TPU runtime available to your account.
2. Execute the cells in order. Start with `PROFILE = "quick"`.
3. Save the exported ZIP for that runtime.
4. Repeat in another runtime. Use the final comparison cell to load CSV results from multiple sessions.

Official references:

- [Colab FAQ](https://research.google.com/colaboratory/faq.html)
- [JAX installation and accelerator support](https://docs.jax.dev/en/latest/installation.html)
- [JAX benchmarking methodology](https://docs.jax.dev/en/latest/benchmarking.html)


In [ ]:
import os, sys, json, time, platform, subprocess, warnings, math, shutil
import datetime
from pathlib import Path
from functools import partial
from collections import Counter
import importlib.metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import jax
    import jax.numpy as jnp
except Exception as exc:
    raise RuntimeError(
        "JAX could not be imported. Consult https://docs.jax.dev/en/latest/installation.html "
        "and install the backend appropriate for your Colab runtime, then restart."
    ) from exc

# Prefer accurate float32 matmul for cross-device comparisons.
jax.config.update("jax_default_matmul_precision", "highest")

DEVICES = jax.devices()
DEVICE = DEVICES[0]
BACKEND = DEVICE.platform
DEVICE_NAME = getattr(DEVICE, "device_kind", str(DEVICE))
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

try:
    JAXLIB_VERSION = importlib.metadata.version("jaxlib")
except Exception:
    JAXLIB_VERSION = "unknown"

def command_output(command):
    try:
        r = subprocess.run(command, capture_output=True, text=True, timeout=10)
        return r.stdout.strip() if r.returncode == 0 else ""
    except Exception:
        return ""

GPU_INFO = command_output([
    "nvidia-smi", "--query-gpu=name,memory.total,driver_version",
    "--format=csv,noheader"
])
TPU_ENV = {k: v for k, v in os.environ.items()
           if k in ("TPU_NAME", "TPU_WORKER_ID", "TPU_WORKER_HOSTNAMES",
                    "TPU_ACCELERATOR_TYPE")}

METADATA = {
    "run_id": RUN_ID,
    "backend": BACKEND,
    "device_kind": DEVICE_NAME,
    "jax_version": jax.__version__,
    "jaxlib_version": JAXLIB_VERSION,
    "python": sys.version,
    "platform": platform.platform(),
    "local_device_count": len(jax.local_devices()),
    "devices": [str(x) for x in DEVICES],
    "nvidia_smi": GPU_INFO,
    "tpu_environment": TPU_ENV,
    "matmul_precision": "highest",
}

print(json.dumps(METADATA, indent=2))
if BACKEND == "cpu":
    warnings.warn(
        "JAX is using CPU. If you selected GPU/TPU in Colab, "
        "check the accelerator runtime and JAX backend installation."
    )
print("\nBenchmarks target ONE device:", DEVICE_NAME)


In [ ]:
#@title Benchmark configuration
PROFILE = "quick" #@param ["quick", "extended"]
SEED = 2026 #@param {type:"integer"}
D = 512 #@param {type:"integer"}
BETA = 16.0 #@param {type:"number"}

N_VALUES = [128, 512] if PROFILE == "quick" else [128, 512, 2048]
Q = 128 if PROFILE == "quick" else 256
REPEATS = 8 if PROFILE == "quick" else 20
WARMUP = 2 if PROFILE == "quick" else 4
NOISE_LEVELS = [0.0, 0.10, 0.25, 0.40]

FLY_DIM = 2048
FLY_FANIN = 16
FLY_K = 64

VSA_DIM = 4096
VSA_N = 160

SDM_DIM = 256
SDM_M = 1024
SDM_K = 16

RUN_SDM = True
RUN_FAULTS = True

RESULTS = []

print("Profile:", PROFILE)
print("Storage sizes:", N_VALUES)
print("Embedding dimension:", D)
print("Query batch:", Q)
print("Warmup/repetitions:", WARMUP, REPEATS)
print("Backend:", BACKEND, DEVICE_NAME)


In [ ]:
# Core deterministic data, algorithms, and timing helpers.

def bipolar(n, d, seed):
    rng = np.random.default_rng(seed)
    return (2 * rng.integers(0, 2, size=(n, d)) - 1).astype(np.float32)

def corrupt_bits(x, probability, rng):
    flip = rng.random(x.shape) < probability
    return np.where(flip, -x, x).astype(np.float32)

def to_device(x):
    return jax.device_put(np.asarray(x), DEVICE)

def synchronize(tree):
    for leaf in jax.tree_util.tree_leaves(tree):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()

def benchmark(fn, *args, batch_size, repeats=None, warmup=None):
    repeats = REPEATS if repeats is None else repeats
    warmup = WARMUP if warmup is None else warmup
    for _ in range(warmup):
        synchronize(fn(*args))
    samples = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        synchronize(fn(*args))
        samples.append((time.perf_counter() - t0) * 1000)
    p50 = float(np.percentile(samples, 50))
    return {
        "batch_p50_ms": p50,
        "batch_p95_ms": float(np.percentile(samples, 95)),
        "batch_p99_ms": float(np.percentile(samples, 99)),
        "amortized_us_per_query": p50 * 1000 / batch_size,
        "throughput_qps": batch_size * 1000 / p50 if p50 > 0 else np.nan,
        "timing_repeats": repeats,
    }

def record(experiment, method, n, d, noise=np.nan, **kwargs):
    RESULTS.append({
        "run_id": RUN_ID,
        "backend": BACKEND,
        "device_kind": DEVICE_NAME,
        "experiment": experiment,
        "method": method,
        "n": n,
        "d": d,
        "noise": noise,
        "query_batch": kwargs.pop("query_batch", Q),
        **kwargs
    })

@jax.jit
def exact_top1(memory, queries):
    return jnp.argmax(queries @ memory.T, axis=1)

@jax.jit
def exact_top10(memory, queries):
    return jax.lax.top_k(queries @ memory.T, 10)[1]

@jax.jit
def hopfield_cleanup(memory, queries, beta):
    # Content-addressable attention, followed by cleanup against stored patterns.
    scores = (queries @ memory.T) / memory.shape[1]
    weights = jax.nn.softmax(beta * scores, axis=-1)
    reconstructed = weights @ memory
    return jnp.argmax(reconstructed @ memory.T, axis=1)

print("Core functions defined.")


## 1. Noisy-key associative retrieval

The report found strong noisy-key retrieval for both exact flat-vector search and Modern Hopfield memory. This experiment compares them with equal-norm bipolar patterns, controlled random bit flips, and the same queries.

**Interpretation:** An exact top-1 match and a Hopfield cleanup top-1 match are not equivalent algorithms, but they solve the same retrieval task. If both achieve similar accuracy, compare latency, scalability, memory, and robustness rather than claiming an accuracy advantage.

Timing is steady-state **device-resident batch timing** after JIT warmup. It excludes host transfers and index construction.


In [ ]:
for n in N_VALUES:
    memory_np = bipolar(n, D, SEED + n)
    memory = to_device(memory_np)

    for noise in NOISE_LEVELS:
        rng = np.random.default_rng(SEED + 10000 + n + round(noise * 1000))
        targets = rng.integers(0, n, size=Q)
        query_np = corrupt_bits(memory_np[targets], noise, rng)
        queries = to_device(query_np)

        methods = [
            ("exact_flat", lambda m, q: exact_top1(m, q)),
            ("hopfield_cleanup", lambda m, q: hopfield_cleanup(m, q, BETA)),
        ]
        for name, fn in methods:
            predictions = np.asarray(fn(memory, queries))
            metrics = benchmark(fn, memory, queries, batch_size=Q)
            record(
                "noisy_recall", name, n, D, noise,
                recall_at_1=float(np.mean(predictions == targets)),
                index_bytes=int(memory_np.nbytes),
                **metrics
            )

retrieval_df = pd.DataFrame(RESULTS)
display(retrieval_df[retrieval_df.experiment == "noisy_recall"].round(4))

# Transfer-inclusive reference: host query -> accelerator -> host result.
# The host query is copied anew on each invocation to avoid buffer reuse.
n = N_VALUES[-1]
memory_np = bipolar(n, D, SEED + n)
memory = to_device(memory_np)
rng = np.random.default_rng(SEED + 424242)
targets = rng.integers(0, n, size=Q)
query_np = corrupt_bits(memory_np[targets], 0.25, rng)

def transfer_inclusive_exact():
    q = jax.device_put(query_np.copy(), DEVICE)
    return np.asarray(exact_top1(memory, q))

metrics = benchmark(transfer_inclusive_exact, batch_size=Q)
record("transfer_inclusive", "exact_flat_host_to_host", n, D, 0.25,
       recall_at_1=float(np.mean(transfer_inclusive_exact() == targets)),
       index_bytes=int(memory_np.nbytes), **metrics)
print("Transfer-inclusive benchmark:", metrics)


## 2. FlyHash-style sparse random projection

A sparse random projection maps bipolar input patterns into a larger space. A winner-take-all operation retains the largest `FLY_K` activations. Similarity is measured by overlap between the resulting binary codes.

**This is a FlyHash-style proxy, not a byte-for-byte recreation of the report's `FlyHashCoder`.** It uses dense float32 matrix multiplication for accelerator compatibility. A production packed-bit implementation could have very different latency and storage costs.

The notebook reports both the dense float32 implementation size and the theoretical bit-packed size. Projection-matrix storage is separate from encoded-record storage.


In [ ]:
rng = np.random.default_rng(SEED + 3000)
projection_np = np.zeros((D, FLY_DIM), dtype=np.float32)

for column in range(FLY_DIM):
    chosen = rng.choice(D, size=FLY_FANIN, replace=False)
    projection_np[chosen, column] = (
        rng.choice([-1.0, 1.0], size=FLY_FANIN) / np.sqrt(FLY_FANIN)
    )

projection = to_device(projection_np)

@partial(jax.jit, static_argnames=("k",))
def fly_encode(x, proj, k):
    activations = x @ proj
    _, winners = jax.lax.top_k(activations, k)
    rows = jnp.arange(x.shape[0])[:, None]
    return jnp.zeros(
        (x.shape[0], proj.shape[1]), dtype=jnp.float32
    ).at[rows, winners].set(1.0)

@jax.jit
def fly_search_top1(codes, queries, proj):
    query_codes = fly_encode(queries, proj, FLY_K)
    return jnp.argmax(query_codes @ codes.T, axis=1)

@jax.jit
def fly_search_top10(codes, queries, proj):
    query_codes = fly_encode(queries, proj, FLY_K)
    return jax.lax.top_k(query_codes @ codes.T, 10)[1]

for n in N_VALUES:
    memory_np = bipolar(n, D, SEED + n)
    memory = to_device(memory_np)
    memory_codes = fly_encode(memory, projection, FLY_K)
    synchronize(memory_codes)

    build_metrics = benchmark(
        lambda: fly_encode(memory, projection, FLY_K),
        batch_size=n
    )
    record("index_build", "flyhash_proxy", n, FLY_DIM,
           query_batch=n, index_build_p50_ms=build_metrics["batch_p50_ms"],
           index_bytes=n * FLY_DIM * 4,
           ideal_packed_code_bytes=math.ceil(n * FLY_DIM / 8),
           projection_dense_bytes=int(projection_np.nbytes))

    for noise in NOISE_LEVELS:
        rng = np.random.default_rng(SEED + 10000 + n + round(noise * 1000))
        targets = rng.integers(0, n, size=Q)
        queries = to_device(corrupt_bits(memory_np[targets], noise, rng))

        top1 = np.asarray(fly_search_top1(memory_codes, queries, projection))
        top10 = np.asarray(fly_search_top10(memory_codes, queries, projection))
        metrics = benchmark(
            fly_search_top1, memory_codes, queries, projection, batch_size=Q
        )
        record(
            "noisy_recall", "flyhash_proxy", n, D, noise,
            recall_at_1=float(np.mean(top1 == targets)),
            recall_at_10=float(np.mean(np.any(top10 == targets[:, None], axis=1))),
            index_bytes=n * FLY_DIM * 4,
            ideal_packed_code_bytes=math.ceil(n * FLY_DIM / 8),
            projection_dense_bytes=int(projection_np.nbytes),
            **metrics
        )

display(pd.DataFrame(RESULTS).query(
    "experiment == 'noisy_recall' and method == 'flyhash_proxy'"
).round(4))


## 3. Role–filler VSA: partial structured-record retrieval

Construct synthetic five-field records: department, role, location, team, and unique ID. Bind field-role hypervectors to filler hypervectors and bundle them into a bipolar record. Retrieve using the first three known fields.

We explicitly stratify **unique** versus **ambiguous** partial-cue tuples. A cue shared by several records cannot uniquely identify the original ID without additional information. The exact tuple-index oracle reports the highest possible top-1 accuracy under deterministic selection of one candidate per identical cue.

This is a VSA retrieval proxy. It does not implement the report's exact VSA-plus-Hopfield cleanup pipeline.


In [ ]:
rng = np.random.default_rng(SEED + 4000)
field_sizes = [6, 8, 6, 10, VSA_N]
fields = np.stack([
    rng.integers(0, size, size=VSA_N)
    for size in field_sizes[:4]
] + [np.arange(VSA_N)], axis=1)

roles = bipolar(5, VSA_DIM, SEED + 4001)
fillers = [
    bipolar(size, VSA_DIM, SEED + 4100 + j)
    for j, size in enumerate(field_sizes)
]
bound = [
    roles[j][None, :] * fillers[j][fields[:, j]]
    for j in range(5)
]
record_codes_np = np.where(
    np.sum(np.stack(bound, axis=0), axis=0) >= 0, 1.0, -1.0
).astype(np.float32)
cue_codes_np = np.where(
    np.sum(np.stack(bound[:3], axis=0), axis=0) >= 0, 1.0, -1.0
).astype(np.float32)

record_codes = to_device(record_codes_np)
cue_codes = to_device(cue_codes_np)

@jax.jit
def vsa_top1(records, cues):
    return jnp.argmax(cues @ records.T, axis=1)

@jax.jit
def vsa_top10(records, cues):
    return jax.lax.top_k(cues @ records.T, 10)[1]

pred = np.asarray(vsa_top1(record_codes, cue_codes))
top10 = np.asarray(vsa_top10(record_codes, cue_codes))
truth = np.arange(VSA_N)

cue_tuples = [tuple(row) for row in fields[:, :3]]
counts = Counter(cue_tuples)
unique_mask = np.array([counts[t] == 1 for t in cue_tuples])
ambiguous_mask = ~unique_mask
oracle_top1_bound = len(counts) / VSA_N

metrics = benchmark(vsa_top1, record_codes, cue_codes, batch_size=VSA_N)
record(
    "structured_completion", "vsa_proxy", VSA_N, VSA_DIM,
    query_batch=VSA_N,
    recall_at_1=float(np.mean(pred == truth)),
    recall_at_10=float(np.mean(np.any(top10 == truth[:, None], axis=1))),
    unique_cue_fraction=float(unique_mask.mean()),
    unique_cue_recall_at_1=(
        float(np.mean(pred[unique_mask] == truth[unique_mask]))
        if unique_mask.any() else np.nan
    ),
    ambiguous_cue_recall_at_1=(
        float(np.mean(pred[ambiguous_mask] == truth[ambiguous_mask]))
        if ambiguous_mask.any() else np.nan
    ),
    exact_tuple_top1_oracle=oracle_top1_bound,
    index_bytes=int(record_codes_np.nbytes),
    **metrics
)
print("Unique cues:", int(unique_mask.sum()), "/", VSA_N)
print("Ambiguous cues:", int(ambiguous_mask.sum()), "/", VSA_N)
print("VSA top-1:", round(float(np.mean(pred == truth)), 4))
print("VSA top-10:", round(float(np.mean(np.any(top10 == truth[:, None], axis=1))), 4))
print("Exact tuple top-1 oracle:", round(oracle_top1_bound, 4))
print("Unique-cue top-1:", round(float(np.mean(pred[unique_mask] == truth[unique_mask])), 4)
      if unique_mask.any() else "N/A")


## 4. Sparse distributed memory capacity

The SDM proxy uses `SDM_M` random bipolar hard locations. Each key activates its `SDM_K` nearest hard locations; values are accumulated in signed float32 counters. Readout sums activated counters and takes the sign.

This is **fixed-k SDM**. The original report used a particular address-activation configuration and write/read implementation; these numbers must not be compared directly with its published SDM capacity table. The vectorized dense construction is also not a benchmark of practical incremental sparse writes.


In [ ]:
@partial(jax.jit, static_argnames=("k",))
def sdm_locations(keys, addresses, k):
    return jax.lax.top_k(keys @ addresses.T, k)[1]

@partial(jax.jit, static_argnames=("k",))
def sdm_build(keys, values, addresses, k):
    locations = sdm_locations(keys, addresses, k)
    mask = jnp.zeros(
        (keys.shape[0], addresses.shape[0]), dtype=jnp.float32
    ).at[jnp.arange(keys.shape[0])[:, None], locations].set(1.0)
    return mask.T @ values

@partial(jax.jit, static_argnames=("k",))
def sdm_read(queries, addresses, counters, k):
    locations = sdm_locations(queries, addresses, k)
    mask = jnp.zeros(
        (queries.shape[0], addresses.shape[0]), dtype=jnp.float32
    ).at[jnp.arange(queries.shape[0])[:, None], locations].set(1.0)
    decoded = mask @ counters
    return jnp.where(decoded >= 0, 1.0, -1.0)

if RUN_SDM:
    addresses = to_device(bipolar(SDM_M, SDM_DIM, SEED + 5000))
    for n in [50, 100, 200, 400]:
        memory_np = bipolar(n, SDM_DIM, SEED + 5000 + n)
        memory = to_device(memory_np)
        counters = sdm_build(memory, memory, addresses, SDM_K)
        synchronize(counters)

        build_metrics = benchmark(
            lambda: sdm_build(memory, memory, addresses, SDM_K),
            batch_size=n
        )
        record(
            "index_build", "sdm_fixed_k", n, SDM_DIM,
            query_batch=n,
            index_build_p50_ms=build_metrics["batch_p50_ms"],
            index_bytes=SDM_M * SDM_DIM * 4 + SDM_M * SDM_DIM * 4
        )

        for noise in [0.0, 0.15]:
            rng = np.random.default_rng(SEED + n + round(noise * 1000))
            queries = to_device(corrupt_bits(memory_np, noise, rng))
            decoded = np.asarray(sdm_read(queries, addresses, counters, SDM_K))
            metrics = benchmark(
                lambda q: sdm_read(q, addresses, counters, SDM_K),
                queries, batch_size=n
            )
            record(
                "sdm_capacity", "sdm_fixed_k", n, SDM_DIM, noise,
                query_batch=n,
                exact_pattern_recall=float(np.mean(np.all(decoded == memory_np, axis=1))),
                bit_error_rate=float(np.mean(decoded != memory_np)),
                index_bytes=SDM_M * SDM_DIM * 8,
                **metrics
            )

    display(pd.DataFrame(RESULTS).query("experiment == 'sdm_capacity'").round(4))
else:
    print("SDM experiment disabled.")


## 5. Out-of-distribution rejection

A nearest-neighbor system always produces a best match, including for an unseen key. We calibrate a maximum-similarity rejection threshold using one set of known noisy queries, then evaluate true-positive and false-positive rates on independent known/OOD queries.

The test uses synthetic independent random OOD keys, not real semantic distribution shifts or adversarial chimeras. A successful result is therefore only a baseline sanity check.

Threshold: the 5th percentile of calibration-set known-query similarity (nominal 95% calibration TPR). AUROC is estimated from pairwise known-versus-OOD score comparisons.


In [ ]:
@jax.jit
def max_similarity(memory, queries):
    return jnp.max((queries @ memory.T) / memory.shape[1], axis=1)

ood_n = 256 if PROFILE == "quick" else 512
ood_q = 128 if PROFILE == "quick" else 256
memory_np = bipolar(ood_n, D, SEED + 6000)
memory = to_device(memory_np)
rng = np.random.default_rng(SEED + 6001)

cal_ids = rng.integers(0, ood_n, size=ood_q)
test_ids = rng.integers(0, ood_n, size=ood_q)
cal_queries = to_device(corrupt_bits(memory_np[cal_ids], 0.25, rng))
test_queries = to_device(corrupt_bits(memory_np[test_ids], 0.25, rng))
ood_queries = to_device(bipolar(ood_q, D, SEED + 6002))

cal_scores = np.asarray(max_similarity(memory, cal_queries))
test_scores = np.asarray(max_similarity(memory, test_queries))
ood_scores = np.asarray(max_similarity(memory, ood_queries))

threshold = float(np.quantile(cal_scores, 0.05))
tpr = float(np.mean(test_scores >= threshold))
fpr = float(np.mean(ood_scores >= threshold))
auroc = float(
    np.mean(test_scores[:, None] > ood_scores[None, :]) +
    0.5 * np.mean(test_scores[:, None] == ood_scores[None, :])
)
metrics = benchmark(max_similarity, memory, test_queries, batch_size=ood_q)

record(
    "ood_rejection", "max_cosine_gate", ood_n, D, 0.25,
    query_batch=ood_q,
    auroc=auroc, tpr=tpr, fpr=fpr,
    calibrated_threshold=threshold,
    index_bytes=int(memory_np.nbytes),
    **metrics
)
print(f"Threshold: {threshold:.4f}")
print(f"Independent test TPR: {tpr:.3f}")
print(f"Independent test FPR: {fpr:.3f}")
print(f"AUROC: {auroc:.4f}")


## 6. Hardware fault-injection simulation

Test stored-pattern corruption with:

- **Digital bit flips:** randomly invert stored bipolar components.
- **Analog-like perturbations:** add Gaussian noise to stored components.

The query set remains fixed across severity levels. These are *functional models* of faults. They do not simulate physical conductance distributions, ADC resolution, DAC costs, line resistance, stuck-at defects, write endurance, retention drift, or calibrated FeNAND behavior.


In [ ]:
if RUN_FAULTS:
    n = 256
    rng = np.random.default_rng(SEED + 7000)
    clean_memory = bipolar(n, D, SEED + 7001)
    targets = rng.integers(0, n, size=Q)
    query_np = corrupt_bits(clean_memory[targets], 0.20, rng)
    queries = to_device(query_np)

    conditions = [
        ("bitflip", x) for x in [0.0, 0.001, 0.01, 0.05]
    ] + [
        ("analog_gaussian", x) for x in [0.05, 0.20, 0.50]
    ]

    for fault_mode, severity in conditions:
        if fault_mode == "bitflip":
            faulty_np = corrupt_bits(clean_memory, severity, rng)
        else:
            faulty_np = (
                clean_memory +
                rng.normal(0.0, severity, clean_memory.shape).astype(np.float32)
            )
        faulty_memory = to_device(faulty_np)

        for method, fn in [
            ("exact_flat", lambda m, q: exact_top1(m, q)),
            ("hopfield_cleanup", lambda m, q: hopfield_cleanup(m, q, BETA)),
        ]:
            predictions = np.asarray(fn(faulty_memory, queries))
            metrics = benchmark(fn, faulty_memory, queries, batch_size=Q)
            record(
                "fault_injection", method, n, D,
                fault_mode=fault_mode,
                severity=severity,
                recall_at_1=float(np.mean(predictions == targets)),
                index_bytes=int(faulty_np.nbytes),
                **metrics
            )

    display(pd.DataFrame(RESULTS).query(
        "experiment == 'fault_injection'"
    )[["method", "fault_mode", "severity", "recall_at_1",
       "batch_p50_ms"]].round(4))
else:
    print("Fault injection disabled.")


## 7. Results visualization and interpretation

These figures visualize measurements from the current session only.

**Timing guidance:** Device-resident batch p50/p95/p99 values are estimates from repeated warmed-up runs. For stronger tail-latency claims, increase repetitions, use independently sampled workloads, and measure actual individual request latency in an end-to-end service. Avoid comparing JAX's batched kernel throughput with the report's Python CPU microseconds as if they were the same measurement.


In [ ]:
df = pd.DataFrame(RESULTS)
display(df.round(4))

retrieval = df[df.experiment == "noisy_recall"].copy()
if not retrieval.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    largest_n = max(retrieval.n)
    for method, group in retrieval[retrieval.n == largest_n].groupby("method"):
        group = group.sort_values("noise")
        axes[0].plot(group.noise, group.recall_at_1,
                     marker="o", label=method)
    axes[0].set(
        title=f"Noisy-key recall at N={largest_n}",
        xlabel="Fraction of query bits flipped",
        ylabel="Recall@1",
        ylim=(-0.03, 1.04)
    )
    axes[0].grid(alpha=0.25)
    axes[0].legend()

    latency = retrieval[np.isclose(retrieval.noise, 0.25)]
    for method, group in latency.groupby("method"):
        group = group.sort_values("n")
        axes[1].plot(group.n, group.batch_p50_ms,
                     marker="o", label=method)
    axes[1].set(
        title="Device-resident batch latency at 25% noise",
        xlabel="Number of stored patterns",
        ylabel="Batch p50 (ms)"
    )
    axes[1].grid(alpha=0.25)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

faults = df[df.experiment == "fault_injection"]
if not faults.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, mode in zip(axes, ["bitflip", "analog_gaussian"]):
        part = faults[faults.fault_mode == mode]
        for method, group in part.groupby("method"):
            group = group.sort_values("severity")
            ax.plot(group.severity, group.recall_at_1,
                    marker="o", label=method)
        ax.set(title=mode, xlabel="Fault severity",
               ylabel="Recall@1", ylim=(-0.03, 1.04))
        ax.grid(alpha=0.25)
        ax.legend()
    plt.tight_layout()
    plt.show()

print("Experiment rows:", len(df))
print("Hardware:", DEVICE_NAME, "| Backend:", BACKEND)


## 8. Export reproducible results

Exports include:

- `results.csv`: long-form experimental metrics.
- `results.json`: the same records as JSON.
- `hardware_metadata.json`: actual backend, JAX versions, device details, runtime identifiers.
- `config.json`: parameters used for the run.
- `summary.txt`: notes on methodological limitations.

Each session has a unique UTC `run_id`, enabling later cross-hardware comparisons. Colab's ephemeral filesystem can be cleared when a runtime ends, so **download the ZIP before disconnecting**.


In [ ]:
OUTPUT_DIR = Path("/content/neurostore_results") / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.DataFrame(RESULTS)
df.to_csv(OUTPUT_DIR / "results.csv", index=False)
(OUTPUT_DIR / "results.json").write_text(
    df.to_json(orient="records", indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "hardware_metadata.json").write_text(
    json.dumps(METADATA, indent=2), encoding="utf-8"
)

CONFIG = {
    "profile": PROFILE, "seed": SEED, "dimension": D,
    "n_values": N_VALUES, "queries_per_batch": Q,
    "repeats": REPEATS, "warmup": WARMUP,
    "noise_levels": NOISE_LEVELS, "hopfield_beta": BETA,
    "fly_dim": FLY_DIM, "fly_fanin": FLY_FANIN, "fly_k": FLY_K,
    "vsa_dim": VSA_DIM, "vsa_n": VSA_N,
    "sdm_dim": SDM_DIM, "sdm_locations": SDM_M, "sdm_k": SDM_K,
    "run_sdm": RUN_SDM, "run_faults": RUN_FAULTS,
}
(OUTPUT_DIR / "config.json").write_text(
    json.dumps(CONFIG, indent=2), encoding="utf-8"
)

NOTES = """NeuroStore accelerator concept benchmark.
Independent proxy implementation, not a reproduction of original report code.
One JAX device is benchmarked per session.
Device-resident timings exclude compilation and transfers unless named otherwise.
Batch latency and amortized throughput are not single-request tail latency.
Dense float32 FlyHash storage is not packed-bit storage.
SDM uses fixed-k activation and vectorized construction.
Fault injection is not a calibrated device-physics simulation.
No energy, endurance, hardware area, or FeNAND speedup claims are measured.
"""
(OUTPUT_DIR / "summary.txt").write_text(NOTES, encoding="utf-8")

ZIP_PATH = shutil.make_archive(
    f"/content/NeuroStore_{RUN_ID}_{BACKEND}", "zip",
    root_dir=OUTPUT_DIR
)
print("Saved results to:", OUTPUT_DIR)
print("ZIP:", ZIP_PATH)

#@title Download this runtime's result bundle
AUTO_DOWNLOAD = False #@param {type:"boolean"}
if AUTO_DOWNLOAD:
    from google.colab import files
    files.download(ZIP_PATH)
else:
    print("Set AUTO_DOWNLOAD=True and rerun this cell to download.")


## 9. Compare GPU and TPU sessions

After separately benchmarking different Colab runtimes, export each run's ZIP. Extract the `results.csv` files and upload them here.

Set `COMPARE_UPLOADS = True` and execute the next cell. Upload only CSVs from comparable configurations (same profile, N, D, noise, query batch, and JAX precision). Different runtime versions and available memory may affect comparability.

**Decision guidance:** Evaluate accuracy, batch throughput, latency, memory footprint, and implementation complexity. Do not infer energy efficiency, FeNAND benefits, or cost per query without independent hardware measurements.


In [ ]:
#@title Optional multi-runtime comparison
COMPARE_UPLOADS = False #@param {type:"boolean"}

if COMPARE_UPLOADS:
    from google.colab import files
    import io
    uploaded = files.upload()
    frames = []
    for filename, data in uploaded.items():
        if filename.lower().endswith(".csv"):
            frame = pd.read_csv(io.BytesIO(data))
            frame["source_file"] = filename
            frames.append(frame)

    if frames:
        combined = pd.concat(frames, ignore_index=True)
        key_cols = [
            "backend", "device_kind", "method", "n", "d", "noise", "query_batch"
        ]
        selected = combined[
            combined.experiment == "noisy_recall"
        ][key_cols + ["recall_at_1", "batch_p50_ms", "throughput_qps"]]
        display(selected.sort_values(["method", "n", "noise", "device_kind"]))
        selected.to_csv("/content/neurostore_cross_hardware_comparison.csv",
                        index=False)
        print("Saved /content/neurostore_cross_hardware_comparison.csv")
    else:
        print("No CSV files uploaded.")
else:
    print("When ready, set COMPARE_UPLOADS=True and run this cell.")
